---

## Chapter 2: Gathering Your Data

In this chapter, we'll pull 4 years of annual financial statements for Persistent Systems Limited using Yahoo Finance API.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

# Persistent Systems trades as PERSISTENT on NSE
ticker = yf.Ticker("PERSISTENT.NS")

# Pull 4 years of annual financials (gives us trend, not just snapshot)
income_stmt  = ticker.financials          # Income Statement (annual)
balance_sheet = ticker.balance_sheet      # Balance Sheet (annual)
cash_flow    = ticker.cashflow            # Cash Flow Statement (annual)

print("✓ Successfully fetched financial data for PERSISTENT.NS")

### Sanity Check: Inspect Data Structure

Always verify your data structure before proceeding with analysis.

In [ ]:
print("=== INCOME STATEMENT COLUMNS ===")
print(income_stmt.columns.tolist())

print("\n=== INCOME STATEMENT ROWS ===")
print(income_stmt.index.tolist())

print("\n=== BALANCE SHEET COLUMNS ===")
print(balance_sheet.columns.tolist())

print("\n=== CASH FLOW STATEMENT COLUMNS ===")
print(cash_flow.columns.tolist())

In [ ]:
# Save raw imported data to CSV for reference
import os

os.makedirs('data', exist_ok=True)

# Export raw statements (transposed for readability)
income_stmt.T.to_csv('data/01_Income_Statement_Raw.csv')
balance_sheet.T.to_csv('data/02_Balance_Sheet_Raw.csv')
cash_flow.T.to_csv('data/03_Cash_Flow_Statement_Raw.csv')

print("✅ Raw data exported to data/ folder:")
print("   • 01_Income_Statement_Raw.csv")
print("   • 02_Balance_Sheet_Raw.csv")
print("   • 03_Cash_Flow_Statement_Raw.csv")

### Save Raw Data to CSV

Export the raw imported financial statements to CSV files for reference and data backup.

---

## Chapter 3: Income Statement

Build a clean, analyst-friendly income statement with key metrics and margins. yFinance returns years as columns, so we'll transpose to make years rows for easier reading.

In [ ]:
# Transpose: convert years from columns to rows (more readable format)
is_raw = income_stmt.T

# Extract key line items that equity analysts care about
income_statement = pd.DataFrame({
    'Total Revenue'        : is_raw.get('Total Revenue',        pd.Series()),
    'Cost of Revenue'      : is_raw.get('Cost Of Revenue',      pd.Series()),
    'Gross Profit'         : is_raw.get('Gross Profit',         pd.Series()),
    'Operating Expense'    : is_raw.get('Operating Expense',    pd.Series()),
    'EBIT'                 : is_raw.get('EBIT',                 pd.Series()),
    'Interest Expense'     : is_raw.get('Interest Expense',     pd.Series()),
    'Pretax Income'        : is_raw.get('Pretax Income',        pd.Series()),
    'Tax Provision'        : is_raw.get('Tax Provision',        pd.Series()),
    'Net Income'           : is_raw.get('Net Income',           pd.Series()),
})

# Convert from raw rupees to Crores (÷ 10,000,000)
# Standard reporting unit for Indian companies
income_statement = income_statement / 1e7

print("✓ Income Statement extracted and converted to ₹ Crores")

### Calculate Key Profitability Margins

In [ ]:
# Calculate key profitability margins
income_statement['Gross Margin %'] = (
    income_statement['Gross Profit'] / income_statement['Total Revenue'] * 100
).round(2)

income_statement['Operating Margin %'] = (
    income_statement['EBIT'] / income_statement['Total Revenue'] * 100
).round(2)

income_statement['Net Margin %'] = (
    income_statement['Net Income'] / income_statement['Total Revenue'] * 100
).round(2)

# Sort chronologically: oldest year first (left to right progression)
income_statement = income_statement.sort_index()

print("✓ Key margins calculated")

### Display Formatted Income Statement

In [ ]:
print("\n" + "="*80)
print("PERSISTENT SYSTEMS — INCOME STATEMENT (₹ Crores)")
print("="*80)
print(income_statement.round(2).to_string())
print("="*80)

# Summary statistics
print("\n--- PROFITABILITY TRENDS ---")
margin_cols = ['Gross Margin %', 'Operating Margin %', 'Net Margin %']
print(income_statement[margin_cols].to_string())

---

## Chapter 4: Balance Sheet

Extract and analyze the balance sheet data with key liquidity and solvency ratios used by equity analysts.

In [ ]:
# Transpose and sort balance sheet data by date (oldest first)
bs_raw = balance_sheet.T.sort_index()

# Extract key balance sheet components
balance = pd.DataFrame({
    # Assets
    'Cash & Equivalents'   : bs_raw.get('Cash And Cash Equivalents',         pd.Series()),
    'Total Current Assets' : bs_raw.get('Current Assets',                    pd.Series()),
    'Total Assets'         : bs_raw.get('Total Assets',                      pd.Series()),
    
    # Liabilities
    'Total Current Liab'   : bs_raw.get('Current Liabilities',               pd.Series()),
    'Total Liabilities'    : bs_raw.get('Total Liabilities Net Minority Interest', pd.Series()),
    'Long Term Debt'       : bs_raw.get('Long Term Debt',                    pd.Series()),
    
    # Equity
    'Total Equity'         : bs_raw.get('Stockholders Equity',               pd.Series()),
    'Retained Earnings'    : bs_raw.get('Retained Earnings',                 pd.Series()),
}) / 1e7  # Convert to Crores

print("✓ Balance Sheet extracted and converted to ₹ Crores")

### Calculate Key Financial Ratios

In [ ]:
# Calculate liquidity and solvency ratios
balance['Current Ratio']    = (
    balance['Total Current Assets'] / balance['Total Current Liab']
).round(2)

balance['Debt-to-Equity']   = (
    balance['Total Liabilities'] / balance['Total Equity']
).round(2)

balance['Equity Ratio']     = (
    balance['Total Equity'] / balance['Total Assets'] * 100
).round(2)

print("✓ Key financial ratios calculated")

### Display Formatted Balance Sheet

In [ ]:
print("\n" + "="*80)
print("PERSISTENT SYSTEMS — BALANCE SHEET (₹ Crores)")
print("="*80)
print(balance.round(2).to_string())
print("="*80)

# Summary statistics
print("\n--- KEY BALANCE SHEET RATIOS ---")
ratio_cols = ['Current Ratio', 'Debt-to-Equity', 'Equity Ratio']
print(balance[ratio_cols].to_string())

---

## Chapter 5: Cash Flow Statement

Analyze the cash flow statement - the truth teller of financial health. This shows where real cash actually comes from and goes to, unlike accounting profits.

In [ ]:
# Transpose and sort cash flow data by date (oldest first)
cf_raw = cash_flow.T.sort_index()

# Extract key cash flow components
cashflow = pd.DataFrame({
    'Operating Cash Flow'  : cf_raw.get('Operating Cash Flow',               pd.Series()),
    'Capital Expenditure'  : cf_raw.get('Capital Expenditure',               pd.Series()),
    'Investing Cash Flow'  : cf_raw.get('Investing Cash Flow',               pd.Series()),
    'Financing Cash Flow'  : cf_raw.get('Financing Cash Flow',               pd.Series()),
    'Free Cash Flow'       : cf_raw.get('Free Cash Flow',                    pd.Series()),
}) / 1e7  # Convert to Crores

print("✓ Cash Flow Statement extracted and converted to ₹ Crores")

### Calculate Cash Quality Metrics

In [ ]:
# FCF Margin — how much of every revenue rupee becomes free cash
cashflow['FCF Margin %'] = (
    cashflow['Free Cash Flow'] / income_statement['Total Revenue'] * 100
).round(2)

# Cash Conversion Ratio — is profit turning into real cash?
# Ideal: > 1.0 means more cash than accounting profit (excellent quality)
cashflow['Cash Conversion'] = (
    cashflow['Operating Cash Flow'] / income_statement['Net Income']
).round(2)

print("✓ Cash quality metrics calculated")

### Display Formatted Cash Flow Statement

In [ ]:
print("\n" + "="*80)
print("PERSISTENT SYSTEMS — CASH FLOW STATEMENT (₹ Crores)")
print("="*80)
print(cashflow.round(2).to_string())
print("="*80)

# Summary statistics
print("\n--- CASH QUALITY METRICS ---")
quality_cols = ['FCF Margin %', 'Cash Conversion']
print(cashflow[quality_cols].to_string())

---

## Chapter 6: Verify the Three-Statement Linkage

The three statements are interconnected. Verify these critical links to ensure data integrity:
- **Link 1**: Net Income (Income Statement) flows to Retained Earnings (Balance Sheet)
- **Link 2**: Depreciation (Income Statement) is added back in Cash Flow Statement
- **Link 3**: Free Cash Flow = Operating Cash Flow - Capital Expenditure

In [ ]:
print("\n" + "="*80)
print("THE THREE-STATEMENT LINKAGE CHECK")
print("="*80 + "\n")

print("Link 1: Net Income → Retained Earnings")
print("Net Income (IS):        ", income_statement['Net Income'].values)
print("Retained Earnings (BS): ", balance['Retained Earnings'].values)
print("→ Retained Earnings should grow roughly by Net Income each year\n")

print("Link 2: Depreciation (IS) → Operating CF adjustment (CFS)")
depr = cf_raw.get('Depreciation And Amortization', pd.Series()) / 1e7
print("Depreciation (non-cash, added back in CFS):", depr.values)
print("→ This bridges the gap between Net Income and Operating Cash Flow\n")

print("Link 3: Free Cash Flow = Operating CF - CapEx")
check_fcf = cashflow['Operating Cash Flow'] + cashflow['Capital Expenditure']
print("Calculated FCF: ", check_fcf.values)
print("Reported FCF:   ", cashflow['Free Cash Flow'].values)
print("→ Verify these match to confirm data integrity")
print("="*80)

---

## Chapter 7: Export to Excel

Export the complete 3-statement financial model to Excel format for distribution, presentation, and further analysis. Each statement gets its own worksheet.

In [ ]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
import os

# Create output directory if it doesn't exist
os.makedirs('output', exist_ok=True)

output_path = 'output/Persistent_3Statement_Model.xlsx'

# Export all three statements to separate sheets
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    income_statement.round(2).to_excel(writer, sheet_name='Income Statement')
    balance.round(2).to_excel(writer, sheet_name='Balance Sheet')
    cashflow.round(2).to_excel(writer, sheet_name='Cash Flow Statement')

print(f"✅ Model exported successfully to {output_path}")
print(f"📊 Sheets created:")
print(f"   • Income Statement (with profitability metrics)")
print(f"   • Balance Sheet (with liquidity & solvency ratios)")
print(f"   • Cash Flow Statement (with cash quality metrics)")

---

## Chapter 8: Analyst Summary

Generate a concise, executive summary of key metrics from the most recent year. This is what an equity analyst would focus on when making investment decisions.

In [ ]:
# Extract latest year data from all three statements
latest = income_statement.iloc[-1]
latest_cf = cashflow.iloc[-1]
latest_bs = balance.iloc[-1]

print("\n" + "="*80)
print("ANALYST'S READ ON PERSISTENT SYSTEMS")
print("="*80 + "\n")

print("📊 PROFITABILITY (Income Statement)")
print(f"   Revenue:          ₹{latest['Total Revenue']:.0f} Cr")
print(f"   Net Profit:       ₹{latest['Net Income']:.0f} Cr")
print(f"   Net Margin:       {latest['Net Margin %']:.1f}%")
print(f"   Gross Margin:     {latest['Gross Margin %']:.1f}%")

print("\n💰 CASH GENERATION (Cash Flow Statement)")
print(f"   Free Cash Flow:   ₹{latest_cf['Free Cash Flow']:.0f} Cr")
print(f"   FCF Margin:       {latest_cf['FCF Margin %']:.1f}%")
print(f"   Cash Conversion:  {latest_cf['Cash Conversion']:.2f}x (> 1.0 = excellent quality)")

print("\n⚖️  FINANCIAL HEALTH (Balance Sheet)")
print(f"   Current Ratio:    {latest_bs['Current Ratio']:.2f}x (> 1.5 = healthy)")
print(f"   Debt-to-Equity:   {latest_bs['Debt-to-Equity']:.2f}x (< 1.0 = conservative)")
print(f"   Equity Ratio:     {latest_bs['Equity Ratio']:.1f}% (higher = stronger)")

print("\n" + "="*80)